# Load modules and define functions
## Imports

In [1]:
import os
import seaborn as sns
import scanpy as sc
import scipy.sparse as sp

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

data_dir         = "/Users/wsun/research/CAT/data/"
results_dir      = "classification_result/"

cell_type = "CD8"

## Read in scRNA-seq data

In [2]:
adata = sc.read_h5ad(data_dir + cell_type + "_combined_filtered.h5ad")

print("Count Data:")
print(adata.X.shape)
print(adata.X[:5,:4])
print()
print("Meta Data:")
print(adata.obs.shape)
pd.set_option('display.max_columns', None)  # show all columns
print(adata.obs[:5])
print()

Count Data:
(371010, 6256)
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5 stored elements and shape (5, 4)>
  Coords	Values
  (0, 0)	1.0
  (1, 0)	1.0
  (2, 3)	2.0
  (3, 3)	1.0
  (4, 3)	1.0

Meta Data:
(371010, 45)
                                        study2  n_genes_by_counts  \
P309-GACACGCGTGGTAACG-1-Liu_2025      Liu_2025               1048   
P309-AGCTTGATCGACAGCC-1-Liu_2025      Liu_2025               1116   
P309-GGGTTGCCAAATACAG-1-Liu_2025      Liu_2025               1308   
P25.ut.AGAGCGAAGCTAGTTC-1-Liu_2022    Liu_2022                967   
CCCAATCTCGTCCGTT-1_PEM6C1-Chow_2023  Chow_2023               2077   

                                     total_counts       TRB_cdr3 TRB_v_gene  \
P309-GACACGCGTGGTAACG-1-Liu_2025           2234.0  CAAADRGQDTQYF     TRBV19   
P309-AGCTTGATCGACAGCC-1-Liu_2025           2935.0  CAAADRGQDTQYF     TRBV19   
P309-GGGTTGCCAAATACAG-1-Liu_2025           2119.0  CAAADRGQDTQYF     TRBV19   
P25.ut.AGAGCGAAGCTAGTTC-1-Liu_2022    

## Read in NN prediction results

In [3]:
# list folders starting with "CD8"
folders = [
    f for f in os.listdir(results_dir)
    if f.startswith(cell_type) and os.path.isdir(os.path.join(results_dir, f))
]

print(f"{cell_type} folders:")
print(folders)

CD8 folders:
['CD8_Zheng_2021_learn_rate_0.001_batch_size_32_cla_weight_1.0_rec_weight_1.0_max_epochs_20', 'CD8_Liu_2025_learn_rate_0.001_batch_size_32_cla_weight_1.0_rec_weight_1.0_max_epochs_20', 'CD8_Chow_2023_learn_rate_0.001_batch_size_32_cla_weight_1.0_rec_weight_1.0_max_epochs_20', 'CD8_Liu_2022_learn_rate_0.001_batch_size_32_cla_weight_1.0_rec_weight_1.0_max_epochs_20', 'CD8_Chen_2024_learn_rate_0.001_batch_size_32_cla_weight_1.0_rec_weight_1.0_max_epochs_20']


In [4]:
file_path = os.path.join(results_dir, folders[0], "predictions_test_data.txt")

df = pd.read_csv(file_path, sep="\t", index_col=0)
print("Predictions DataFrame:")
print(df.head())

Predictions DataFrame:
                                     Prediction  True Label
ACTTGTTGTTAAGACA.66-THCA-Zheng_2021    0.000716           0
CCACCTACATTGCGGC.66-THCA-Zheng_2021    0.004297           0
GACGGCTTCAGTTAGC.66-THCA-Zheng_2021    0.077809           0
GCACTCTAGCTAACTC.66-THCA-Zheng_2021    0.004882           0
GTGCAGCCAGTCTTCC.66-THCA-Zheng_2021    0.059357           0


In [5]:

# list folders starting with "CD8" or "CD4"
folders = [
    f for f in os.listdir(results_dir)
    if f.startswith(cell_type) and os.path.isdir(os.path.join(results_dir, f))
]

frames = []

for folder in folders:
    file_path = os.path.join(results_dir, folder, "predictions_test_data.txt")
    if not os.path.exists(file_path):
        print(f"❓ File not found: {file_path}")
        continue

    df = pd.read_csv(file_path, sep="\t", index_col=0)

    # check for any IDs in df not present in adata.obs
    missing_in_obs = df.index.difference(adata.obs.index)
    if len(missing_in_obs) > 0:
        print(f"⚠️  Skipping {folder}: {len(missing_in_obs)} cell IDs not in adata.obs (showing up to 10):")
        print(list(missing_in_obs[:10]))
        continue

    # (optional) drop any rows not in obs, though above check ensures none
    df = df.loc[df.index.intersection(adata.obs.index)]

    frames.append(df)

if not frames:
    print("⚠️  No valid prediction files were loaded; nothing to merge.")
else:
    # row-wise bind (keep cell IDs as index)
    combined = pd.concat(frames, axis=0)

    # warn on duplicate cell IDs across folders
    dup_mask = combined.index.duplicated(keep=False)
    n_dups = dup_mask.sum()
    if n_dups > 0:
        dup_ids = combined.index[dup_mask].unique()
        print(f"⚠️  {n_dups} duplicate rows detected across folders "
              f"({len(dup_ids)} unique cell IDs). Keeping the first occurrence per cell ID.")
        # keep first occurrence per cell ID
        combined = combined[~combined.index.duplicated(keep="first")]



## Combine NN predictions with meta data

In [6]:
common_ids = combined.index.intersection(adata.obs.index)
missing_in_obs = combined.index.difference(adata.obs.index)
missing_in_df = adata.obs.index.difference(combined.index)
print(f"  - Matching IDs: {len(common_ids)}")
print(f"  - In df but not in adata.obs: {len(missing_in_obs)}")
print(f"  - In adata.obs but not in df: {len(missing_in_df)}")



  - Matching IDs: 371010
  - In df but not in adata.obs: 0
  - In adata.obs but not in df: 0


In [7]:
adata.obs = adata.obs.join(combined, how="left")
print(f"✅ Merged {combined.shape[0]} rows and {combined.shape[1]} columns into adata.obs.")


✅ Merged 371010 rows and 2 columns into adata.obs.


In [8]:
adata.obs.to_csv(f"/Users/wsun/research/CAT/data/{cell_type}_with_NN_predictions.tsv", sep="\t", index=True)
